# Person-Specific Sepsis Detection -- Production Pipeline
**Version:** v2 (`sepsis_detector_v2.py`)

### Architecture
| Phase | Duration | What happens |
|---|---|---|
| **Phase A -- Baseline** | T=0 -> T=200s (5 × 40s windows) | Collect vitals, compute 4-component confidence, lock baseline |
| **Phase B -- Monitoring** | T=200s -> ∞ (every 40s) | Z-scores, anomaly scoring, RF, qSOFA, final score, status |

### Confidence Modes
| Mode | Confidence | Blend |
|---|---|---|
| `LOCKED` | >= 75% | 50% Personal IF + 50% Z-score anomaly |
| `HYBRID` | 60-75% | 60% of LOCKED blend + 40% Pop IF |
| `FALLBACK` | < 60% | 100% Population IF (global standards) |


In [1]:
import json
import time
import logging
from IPython.display import clear_output
from vitals_types import VitalsSample, BASELINE_WINDOWS
from simulator import PatientStreamSimulator
from models_factory import build_population_if, build_random_forest
from sepsis_detector import SepsisDetector

print("Building shared models (population IF + Random Forest)...")
pop_if = build_population_if()
rf     = build_random_forest()
print("Ready.")


Building shared models (population IF + Random Forest)...
Ready.


In [2]:
# 0=normal  1=mild infection  2=sepsis (rapid deterioration)
BASELINE_CONDITION = 0

sim      = PatientStreamSimulator(condition=BASELINE_CONDITION)
detector = SepsisDetector(population_if=pop_if, rf_model=rf)
print(f"Initialised with baseline condition={BASELINE_CONDITION}.")


Initialised with baseline condition=0.


## Phase A -- Baseline Establishment
Collect 5 × 40-second windows. After window 5 the baseline **auto-locks**
and prints the confidence score and mode selection.


In [3]:
print("=== PHASE A: Collecting 5-window Baseline ===\n")
for _ in range(BASELINE_WINDOWS):
    sample = sim.get_next_window()
    result = detector.add_baseline_window(sample)
    if result:
        print("\n--- BASELINE LOCKED ---")
        print(json.dumps({
            "confidence":           result.confidence,
            "mode":                 result.mode,
            "confidence_breakdown": result.confidence_breakdown,
            "baseline_means":       {k: round(v, 2) for k, v in result.baseline_means.items()},
        }, indent=2))


=== PHASE A: Collecting 5-window Baseline ===


--- BASELINE LOCKED ---
{
  "confidence": 100.0,
  "mode": "LOCKED",
  "confidence_breakdown": {
    "Stability": 100.0,
    "Consistency": 100.0,
    "Activity": 100.0,
    "Variability": 100.0
  },
  "baseline_means": {
    "hr": 75.33,
    "rr": 14.27,
    "spo2": 97.91,
    "temp": 36.76,
    "movement": 12.39,
    "hrv": 46.16,
    "rrv": 16.28
  }
}


## Phase B -- Real-Time Monitoring
Runs 20 windows of simulated monitoring (condition=2 / rapid sepsis).
Set `DEMO_WINDOWS = None` for an infinite loop.


In [4]:
DEMO_WINDOWS = 20   # None = infinite loop
sim.set_condition(2)

print("=== PHASE B: Real-Time Monitoring (condition=2 / sepsis) ===\n")
idx = 0
try:
    while DEMO_WINDOWS is None or idx < DEMO_WINDOWS:
        out = detector.process_monitoring_window(sim.get_next_window())
        clear_output(wait=True)
        print(f"Window {out['window_number']} | Status: {out['status']} | Phase: {out['sepsis_phase']}")
        print(f"Score: {out['final_score']:.3f} | Conf: {out['baseline_confidence']:.1f}% [{out['baseline_state']}]")
        print(f"qSOFA: {out['qsofa_score']} | HRV collapse: {out['hrv_collapse_severity']:.2f} | Lactate: {out['lactate_proxy']:.2f}")
        print()
        print(json.dumps(out, indent=2))
        time.sleep(0.8)
        idx += 1
except KeyboardInterrupt:
    print("\nMonitoring stopped.")


Window 25 | Status: HIGH_RISK | Phase: PHASE_2_INTERMEDIATE
Score: 0.511 | Conf: 100.0% [LOCKED]
qSOFA: 3 | HRV collapse: 0.34 | Lactate: 1.00

{
  "phase": "MONITORING",
  "window_number": 25,
  "timestamp": "2026-04-11T00:45:56.459061",
  "baseline_state": "LOCKED",
  "baseline_confidence": 100.0,
  "baseline_confidence_breakdown": {
    "Stability": 100.0,
    "Consistency": 100.0,
    "Activity": 100.0,
    "Variability": 100.0
  },
  "drift_from_locked": {
    "hr": 3.5334,
    "rr": 0.9515,
    "spo2": -0.8293,
    "temp": 0.2036,
    "movement": -0.7096,
    "hrv": -2.7987,
    "rrv": -0.9497
  },
  "artifact_contaminated": false,
  "derivatives_available": true,
  "temp_bidirectional_flag": true,
  "vitals_current": {
    "timestamp": "2026-04-11T00:45:56.459061",
    "hr": 120.14,
    "rr": 26.05,
    "spo2": 87.8,
    "temp": 39.24,
    "movement": 8.5,
    "hrv": 11.1,
    "rrv": 5.1,
    "label": 2
  },
  "z_scores": {
    "hr": 129.656,
    "rr": 68.349,
    "spo2": -70.45

## Tuning Confidence Thresholds

Edit the constants at the top of `sepsis_detector_v2.py`:
```python
CONFIDENCE_HIGH = 75.0   # >= this -> LOCKED
CONFIDENCE_MID  = 60.0   # < this -> FALLBACK; between -> HYBRID
```

Run formal test suite to validate all 6 test cases:
```bash
python test_sepsis_v2.py
```
